<a href="https://colab.research.google.com/github/Snowbear65/CS-335-AI/blob/main/chatbot_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q uninstall -y sympy
!pip -q install sympy==1.11.1

In [ ]:
# Basic Knowledge-Base Chatbot
# - Embeds a small knowledge base with SentenceTransformers
# - Retrieves top-k relevant facts using cosine similarity
# - Generates an answer with FLAN-T5 (using model/tokenizer directly for maximum compatibility)

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -----------------------------
# 1) Knowledge base
# -----------------------------
knowledge_base = [
    "CS 335 is a class about AI and how to use it effectively in the real-world.",
    "Professor Omeed has had a cat since 2021; his name is Dino. Dino is 5 years old now.",
    "Professor Omeed has worked for Google from 2022 through 2025"
    "Professor Omeed has been working at CIBC since October 2025"
]

# -----------------------------
# 2) Embedding model + helpers
# -----------------------------
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def normalize(vectors: np.ndarray) -> np.ndarray:
    """Row-wise L2 normalize a 2D numpy array."""
    norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
    return vectors / norms

# Precompute and normalize KB vectors once
kb_vectors = embedder.encode(knowledge_base, convert_to_numpy=True)
kb_vectors = normalize(kb_vectors)

# -----------------------------
# 3) Generator model (robust)
# -----------------------------
MODEL_NAME = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def generate_answer(prompt: str, max_new_tokens: int = 80) -> str:
    """Generate an answer from a seq2seq model (deterministic)."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,     # deterministic
            num_beams=1
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

# -----------------------------
# 4) Retrieval + Chat loop
# -----------------------------
TOP_K = 3
CONF_THRESHOLD = 0.45  # tune (0.35-0.60 typical)
DEBUG_SHOW_RETRIEVAL = False  # set True to print retrieved facts + scores

print("🤖 Chatbot is ready! Type your question or type 'exit' to quit.\n")

while True:
    try:
        question = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n👋 Chatbot says goodbye!")
        break

    if question.lower() in {"exit", "quit"}:
        print("👋 Chatbot says goodbye!")
        break

    if not question:
        print("\n🤖 Chatbot says:\nPlease type a question.\n")
        continue

    # Embed question and normalize
    q_vec = embedder.encode([question], convert_to_numpy=True)
    q_vec = normalize(q_vec)

    # Cosine similarity (dot product of normalized vectors)
    scores = (q_vec @ kb_vectors.T).flatten()

    # Retrieve top-k
    top_indices = np.argsort(scores)[::-1][:TOP_K]
    top_score = float(scores[top_indices[0]])

    if top_score < CONF_THRESHOLD:
        print("\n🤖 Chatbot says:\nI don’t know based on my knowledge base.\n")
        continue

    retrieved = [(knowledge_base[i], float(scores[i])) for i in top_indices]

    if DEBUG_SHOW_RETRIEVAL:
        print("\n[DEBUG] Retrieved facts:")
        for fact, sc in retrieved:
            print(f"  score={sc:.3f} | {fact}")
        print()

    context_block = "\n- " + "\n- ".join([fact for fact, _ in retrieved])

    prompt = (
        "You are a helpful assistant.\n"
        "Answer ONLY using the context below.\n"
        "If the answer is not in the context, reply exactly: "
        "\"I don’t know based on my knowledge base.\"\n\n"
        f"Context:{context_block}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    response = generate_answer(prompt, max_new_tokens=100)
    print("\n🤖 Chatbot says:\n", response, "\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

🤖 Chatbot is ready! Type your question or type 'exit' to quit.

You: Who is omeed

🤖 Chatbot says:
 Professor Omeed 

You: How old is Omeed's cat

🤖 Chatbot says:
 5 years old 

You: Who is Dino

🤖 Chatbot says:
 Professor Omeed 

You: Does omeed have a pet?

🤖 Chatbot says:
 yes 

You: Does omeed have a pet and what is his name?

🤖 Chatbot says:
 Dino 

You: Where does omeed woerk

🤖 Chatbot says:
I don’t know based on my knowledge base.

You: Where does omeed work

🤖 Chatbot says:
 Google 

You: I do not

🤖 Chatbot says:
I don’t know based on my knowledge base.

